In [4]:
import json

with open("merged.json", "r") as f:
    data = json.load(f)

for key in data.keys():
    val = data[key]
    print(f"\n{'='*50}")
    print(f"KEY: '{key}'")
    print(f"类型: {type(val)}")
    
    if isinstance(val, list):
        print(f"列表长度: {len(val)}")
        print(f"第0条类型: {type(val[0])}")
        print(f"第0条内容: {str(val[0])[:300]}")
        if len(val) > 1:
            print(f"第1条内容: {str(val[1])[:300]}")
            
    elif isinstance(val, dict):
        print(f"子keys: {list(val.keys())[:10]}")
        first_subkey = list(val.keys())[0]
        subval = val[first_subkey]
        print(f"data['{key}']['{first_subkey}'] 类型: {type(subval)}")
        print(f"data['{key}']['{first_subkey}'] 内容: {str(subval)[:300]}")
        
    else:
        print(f"内容: {str(val)[:300]}")



KEY: 'all_data_info'
类型: <class 'dict'>
子keys: ['low sensitivity in X-ray detection', 'high sensitivity in X-ray detection', 'low Ion migration activation energy of the crystal', 'high Ion migration activation energy of the crystal', 'high stiffness', 'high carrier mobility', 'high band gap', 'low stiffness', 'low carrier mobility', 'low band gap']
data['all_data_info']['low sensitivity in X-ray detection'] 类型: <class 'list'>
data['all_data_info']['low sensitivity in X-ray detection'] 内容: ['Iron oxides\nDescription: Iron oxides are considered to have low sensitivity in X-ray detection due to their high atomic number (Z) and density, leading to strong absorption of X-rays. This makes it challenging to accurately detect iron oxides using X-ray-based methods.\nSupporting Context: Iron o

KEY: 'all_description'
类型: <class 'dict'>
子keys: ['low sensitivity in X-ray detection', 'high sensitivity in X-ray detection', 'low Ion migration activation energy of the crystal', 'high Ion migration ac

"""
LOO Cross-Validation + k-Sensitivity Analysis + Plotting
=========================================================
使用说明：
  1. 把本文件和 merged.json 放在同一目录
  2. 运行: python LOO_validation_complete.py
  3. 输出文件：
       LOO_TFIDF_stiffness.csv          ← TF-IDF 刚度 LOO 结果
       LOO_TFIDF_activation_energy.csv  ← TF-IDF 激活能 LOO 结果
       LOO_Contriever_stiffness.csv     ← Contriever 刚度 LOO 结果（需网络）
       LOO_Contriever_activation_energy.csv
       Origin_LOO_stiffness.csv         ← Origin 绘图数据（刚度）
       Origin_LOO_activation_energy.csv ← Origin 绘图数据（激活能）
       LOO_k_sensitivity.png            ← 最终图片
"""

In [5]:
import os
import json
import csv
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score
import matplotlib
matplotlib.use('Agg')   # 无界面服务器时不报错
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ==============================================================
# 0. 全局设置
# ==============================================================

MERGED_JSON  = "merged.json"   # 数据文件路径，按需修改
K_MIN, K_MAX, K_STEP = 20, 96, 5
K_RANGE = list(range(K_MIN, K_MAX, K_STEP))

# 两个任务：(任务名, 高类key, 低类key, 随机基线%)
TASKS = [
    (
        "stiffness",
        "high stiffness",
        "low stiffness",
        55.7,          # 低类/(高+低) ≈ 186/334
    ),
    (
        "activation_energy",
        "high Ion migration activation energy of the crystal",
        "low Ion migration activation energy of the crystal",
        53.8,          # 低类/(高+低) ≈ 100/186
    ),
]

In [6]:
# ==============================================================
# 1. 加载数据
# ==============================================================

print("=" * 60)
print("Step 1: Loading merged.json")
print("=" * 60)

with open(MERGED_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

descriptions = data["all_description"]   # 我们只用 description 字段

for task_name, high_key, low_key, baseline in TASKS:
    n_h = len(descriptions[high_key])
    n_l = len(descriptions[low_key])
    print(f"  {task_name}: high={n_h}, low={n_l}, total={n_h+n_l}")

# ==============================================================
# 2. 核心函数
# ==============================================================

def run_loo(embeddings: np.ndarray, labels: np.ndarray, k_range):
    """
    对给定嵌入矩阵和标签执行 LOO 交叉验证，扫描多个 k 值。
    返回 {k: accuracy} 字典。
    """
    N = len(labels)
    results = {}
    for k in k_range:
        preds = []
        for i in range(N):
            # 移除第 i 个样本
            mask = np.ones(N, dtype=bool)
            mask[i] = False
            ref_emb    = embeddings[mask]   # (N-1, d)
            ref_labels = labels[mask]       # (N-1,)
            query      = embeddings[i]      # (d,)

            # 余弦相似度（嵌入已 L2 归一化，点积 = 余弦相似度）
            sims     = ref_emb @ query                  # (N-1,)
            topk_idx = np.argsort(sims)[::-1][:k]       # 最近 k 个
            vote     = ref_labels[topk_idx].sum()        # 高类票数
            preds.append(1 if vote > k / 2 else 0)

        results[k] = accuracy_score(labels, preds)
    return results


def save_loo_csv(fname, k_results):
    """保存 {k: accuracy} 到 CSV"""
    with open(fname, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["k", "LOO_accuracy"])
        for k, acc in k_results.items():
            w.writerow([k, f"{acc:.6f}"])
    print(f"    Saved → {fname}")


def tfidf_embed(texts):
    """TF-IDF 嵌入，L2 归一化"""
    vec = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        stop_words="english",
        sublinear_tf=True,
    )
    mat = vec.fit_transform(texts).toarray().astype(np.float32)
    return normalize(mat, norm="l2")


def contriever_embed(texts, batch_size=32):
    """
    Contriever 嵌入（需要 transformers + 网络 / 本地模型）。
    成功返回 numpy 数组；失败返回 None。
    """
    try:
        import os
        os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
        import torch
        from transformers import AutoTokenizer, AutoModel

        model_name = "akariasai/pes2o_contriever"
        print(f"    Loading Contriever from {os.environ['HF_ENDPOINT']} ...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model     = AutoModel.from_pretrained(model_name)
        model.eval()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model  = model.to(device)
        print(f"    Contriever loaded on {device}")

        all_emb = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc   = tokenizer(
                batch, padding=True, truncation=True,
                max_length=512, return_tensors="pt"
            ).to(device)
            with torch.no_grad():
                out = model(**enc)
            # mean pooling
            mask_exp = enc["attention_mask"].unsqueeze(-1).float()
            emb      = (out[0] * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)
            emb      = torch.nn.functional.normalize(emb, p=2, dim=1)
            all_emb.append(emb.cpu().numpy())
        return np.vstack(all_emb)

    except Exception as e:
        print(f"    ⚠  Contriever unavailable: {e}")
        print(f"    ⚠  Skipping Contriever; TF-IDF results will be used for the plot.")
        return None



Step 1: Loading merged.json
  stiffness: high=148, low=186, total=334
  activation_energy: high=86, low=100, total=186


In [7]:
# ==============================================================
# 3. 对两个任务分别执行 LOO
# ==============================================================

# 存储所有结果，供后续绘图和导出
all_results = {}   # {task_name: {"tfidf": {k:acc}, "contriever": {k:acc}|None}}

for task_name, high_key, low_key, baseline in TASKS:
    print()
    print("=" * 60)
    print(f"Task: {task_name}")
    print("=" * 60)

    high_texts = descriptions[high_key]
    low_texts  = descriptions[low_key]
    texts      = high_texts + low_texts
    labels     = np.array([1] * len(high_texts) + [0] * len(low_texts))
    N          = len(labels)

    # ---- TF-IDF LOO ----
    print(f"  [TF-IDF] Encoding {N} texts ...")
    tfidf_emb  = tfidf_embed(texts)
    print(f"  [TF-IDF] Running LOO (k={K_MIN}–{K_MAX-1}, step={K_STEP}) ...")
    tfidf_res  = run_loo(tfidf_emb, labels, K_RANGE)
    best_k_t   = max(tfidf_res, key=tfidf_res.get)
    best_acc_t = tfidf_res[best_k_t]
    print(f"  [TF-IDF] Best LOO = {best_acc_t*100:.1f}% @ k={best_k_t}  "
          f"(baseline={baseline}%, +{(best_acc_t-baseline/100)*100:.1f} pp)")
    save_loo_csv(f"LOO_TFIDF_{task_name}.csv", tfidf_res)

    # ---- Contriever LOO（可选）----
    print(f"  [Contriever] Encoding {N} texts ...")
    cont_emb = contriever_embed(texts)
    if cont_emb is not None:
        print(f"  [Contriever] Running LOO ...")
        cont_res   = run_loo(cont_emb, labels, K_RANGE)
        best_k_c   = max(cont_res, key=cont_res.get)
        best_acc_c = cont_res[best_k_c]
        print(f"  [Contriever] Best LOO = {best_acc_c*100:.1f}% @ k={best_k_c}  "
              f"(baseline={baseline}%, +{(best_acc_c-baseline/100)*100:.1f} pp)")
        save_loo_csv(f"LOO_Contriever_{task_name}.csv", cont_res)
    else:
        cont_res = None

    all_results[task_name] = {
        "tfidf":      tfidf_res,
        "contriever": cont_res,
        "baseline":   baseline,
        "N_high":     len(high_texts),
        "N_low":      len(low_texts),
    }



Task: stiffness
  [TF-IDF] Encoding 334 texts ...
  [TF-IDF] Running LOO (k=20–95, step=5) ...
  [TF-IDF] Best LOO = 91.9% @ k=35  (baseline=55.7%, +36.2 pp)
    Saved → LOO_TFIDF_stiffness.csv
  [Contriever] Encoding 334 texts ...
    Loading Contriever from https://hf-mirror.com ...


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /akariasai/pes2o_contriever/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x7f1e09e6a6f0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: d57b77a2-0597-4446-ae85-af88c7c7e1f4)')' thrown while requesting HEAD https://huggingface.co/akariasai/pes2o_contriever/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /akariasai/pes2o_contriever/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x7f1e09eace60>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 4ca573db-a469-420a-8089-882de3931d5a)')' thrown while requesting HEAD https://huggingface.co/akariasai/pes2o_contriever/r

    Contriever loaded on cuda
  [Contriever] Running LOO ...
  [Contriever] Best LOO = 76.0% @ k=60  (baseline=55.7%, +20.3 pp)
    Saved → LOO_Contriever_stiffness.csv

Task: activation_energy
  [TF-IDF] Encoding 186 texts ...
  [TF-IDF] Running LOO (k=20–95, step=5) ...
  [TF-IDF] Best LOO = 86.6% @ k=20  (baseline=53.8%, +32.8 pp)
    Saved → LOO_TFIDF_activation_energy.csv
  [Contriever] Encoding 186 texts ...
    Loading Contriever from https://hf-mirror.com ...


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /akariasai/pes2o_contriever/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x7f1e09e748c0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: ba4b8a7e-669a-4761-90a9-ab92f6fb5c67)')' thrown while requesting HEAD https://huggingface.co/akariasai/pes2o_contriever/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /akariasai/pes2o_contriever/resolve/main/tokenizer_config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x7f1e09e4f170>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: a5d39523-cdfd-44bd-849d-6458040119a1)')' thrown while requesting HEAD https://huggingface.co/akariasai/pes2o_contriever/r

    Contriever loaded on cuda
  [Contriever] Running LOO ...
  [Contriever] Best LOO = 77.4% @ k=20  (baseline=53.8%, +23.6 pp)
    Saved → LOO_Contriever_activation_energy.csv


In [ ]:
# ==============================================================
# 4. 导出 Origin 绘图数据
# ==============================================================

print()
print("=" * 60)
print("Step 4: Exporting Origin CSV files")
print("=" * 60)

for task_name, res in all_results.items():
    fname = f"Origin_LOO_{task_name}.csv"
    tfidf_res  = res["tfidf"]
    cont_res   = res["contriever"]
    baseline   = res["baseline"]

    with open(fname, "w", newline="") as f:
        w = csv.writer(f)
        # 表头
        header = ["k", "TF-IDF_LOO_pct"]
        if cont_res:
            header.append("Contriever_LOO_pct")
        header.append("Baseline_pct")
        w.writerow(header)

        # 数据行
        for k in K_RANGE:
            row = [k, round(tfidf_res[k] * 100, 2)]
            if cont_res:
                row.append(round(cont_res[k] * 100, 2))
            row.append(baseline)
            w.writerow(row)

    print(f"  Saved → {fname}")
    print(f"  （Origin 中：导入此文件 → 以 k 为 X 轴，其余列各画一条折线）")


# ==============================================================
# 5. 绘图
# ==============================================================

print()
print("=" * 60)
print("Step 5: Plotting")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

COLORS = {
    "tfidf":      "#E07B39",   # 橙色
    "contriever": "#2E75B6",   # 蓝色
    "baseline":   "#888888",   # 灰色
}

TASK_TITLES = {
    "stiffness":        "Stiffness Task\n(148 high / 186 low, N=334)",
    "activation_energy": "Ion-Migration Activation Energy Task\n(86 high / 100 low, N=186)",
}

for ax, (task_name, res) in zip(axes, all_results.items()):
    tfidf_res  = res["tfidf"]
    cont_res   = res["contriever"]
    baseline   = res["baseline"]

    k_vals       = list(K_RANGE)
    tfidf_accs   = [tfidf_res[k] * 100  for k in k_vals]
    cont_accs    = [cont_res[k]  * 100  for k in k_vals] if cont_res else None

    # ---- TF-IDF 曲线 ----
    ax.plot(k_vals, tfidf_accs,
            "s--", color=COLORS["tfidf"],
            linewidth=2, markersize=6,
            label=f"TF-IDF (best {max(tfidf_accs):.1f}%)")

    # ---- Contriever 曲线 ----
    if cont_accs:
        ax.plot(k_vals, cont_accs,
                "o-", color=COLORS["contriever"],
                linewidth=2.5, markersize=6,
                label=f"Contriever (best {max(cont_accs):.1f}%)")
        # 最优点标注
        best_idx = cont_accs.index(max(cont_accs))
        ax.annotate(
            f" {max(cont_accs):.1f}%",
            xy=(k_vals[best_idx], max(cont_accs)),
            fontsize=9, color=COLORS["contriever"], fontweight="bold",
        )
        # 填充 Contriever 高于基线区域
        ax.fill_between(k_vals, baseline, cont_accs,
                         alpha=0.10, color=COLORS["contriever"])

    # ---- 随机基线 ----
    ax.axhline(baseline, color=COLORS["baseline"],
               linestyle=":", linewidth=2,
               label=f"Random baseline ({baseline}%)")

    # ---- 装饰 ----
    ax.set_xlabel("k  (number of nearest neighbors)", fontsize=12)
    ax.set_ylabel("LOO Classification Accuracy (%)", fontsize=12)
    ax.set_title(TASK_TITLES[task_name], fontsize=11, fontweight="bold")
    ax.set_ylim(40, 100)
    ax.set_xlim(K_MIN - 2, K_MAX + 2)
    ax.legend(fontsize=9, loc="lower left")
    ax.grid(alpha=0.3, linestyle="--")

    # 标注最优 TF-IDF 点
    best_t_idx = tfidf_accs.index(max(tfidf_accs))
    ax.annotate(
        f" {max(tfidf_accs):.1f}%",
        xy=(k_vals[best_t_idx], max(tfidf_accs)),
        fontsize=9, color=COLORS["tfidf"], fontweight="bold",
    )

plt.suptitle(
    "Leave-One-Out Cross-Validation of kNN Reference Text Classifier",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.savefig("LOO_k_sensitivity.png", dpi=300, bbox_inches="tight")
print("  Saved → LOO_k_sensitivity.png")
plt.close()

In [ ]:
# ==============================================================
# 6. 最终汇总（直接复制进回复信）
# ==============================================================

print()
print("=" * 60)
print("FINAL SUMMARY  —  paste into response letter")
print("=" * 60)

for task_name, res in all_results.items():
    tfidf_res = res["tfidf"]
    cont_res  = res["contriever"]
    baseline  = res["baseline"]

    best_k_t  = max(tfidf_res, key=tfidf_res.get)
    best_t    = tfidf_res[best_k_t] * 100

    print(f"\n  Task: {task_name}")
    print(f"    TF-IDF  → best LOO = {best_t:.1f}%  (k={best_k_t}), "
          f"baseline={baseline}%, +{best_t-baseline:.1f} pp")
    if cont_res:
        best_k_c = max(cont_res, key=cont_res.get)
        best_c   = cont_res[best_k_c] * 100
        print(f"    Contriever → best LOO = {best_c:.1f}%  (k={best_k_c}), "
              f"baseline={baseline}%, +{best_c-baseline:.1f} pp")
    else:
        print(f"    Contriever → not available (network issue)")

print()
print("All done! Check the current directory for output files.")
